# Dataset Inspection

High-level structural overview of the entire College Experience Dataset.
Covers file inventory, column inventory, cross-dataset alignment, and student count investigation.

Deep analysis of each data source is in:
- 02_general_EMA_analysis.ipynb
- 03_covid_EMA_analysis.ipynb
- 04_sensing_analysis.ipynb

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

---
## File Inventory

Scans all files in the raw dataset directory and classifies them by role.
Produces a file inventory saved to outputs/tables/.

In [3]:
DATA_DIR = Path("../../data/raw/college_experience_dataset")
print(DATA_DIR)
print(DATA_DIR.exists())

for f in sorted(DATA_DIR.rglob("*")):
    if f.is_file():
        print(f.name)

..\..\data\raw\college_experience_dataset
True
Data Dictionary.csv
demographics.csv
covid_ema.csv
Data Dictionary (COVID EMA).csv
Data Dictionary (general EMA).csv
general_ema.csv
._calllog.csv
._description.txt
calllog.csv
description.txt
._5e1d4d2718012c036ea65bfe3c80c0e7.csv
._description.txt
0ba15aa0582c5e825710d42fe3eb231d.csv
169d3212668c4eed434dafd32e33c946.csv
17fb1517cc9d7e416782bffb8edd9070.csv
1d2263527eed2a54e88d340fb8e55308.csv
212521733ea8eeff63d997dc0213c692.csv
3528757eaf1d15ee6c778eb79cd02de7.csv
35cf1abf179310dc33907d953f590366.csv
45555d3305e255895b11695c179a98cd.csv
557b44c8daf2ddb5d02313f20f2505b9.csv
5e1d4d2718012c036ea65bfe3c80c0e7.csv
73e13f8273906f7f43a077f95ec48e7d.csv
7d2c632a05bbb03ca97555d61be83c41.csv
862f10a8c357e957a59d122077b3a5ad.csv
9ffbe25e279de21de86acd54b6fa60d0.csv
a52b5e80b4c7a8e05f8cc0a16ae4ea9f.csv
a7a320d21e232075512a494c982dc17f.csv
ac70fe1f8115ac361f2023269c011c3e.csv
aeeb186fafcc356f44cae870555f4a0d.csv
bbcad50219ee8a1484b4c6f000eb4e68.csv


In [4]:
all_files = [f for f in sorted(DATA_DIR.rglob("*")) if f.is_file() and not f.name.startswith("._")]
print("Number of files:", len(all_files))

for f in all_files:
    print(f.relative_to(DATA_DIR))

Number of files: 256
Demographics\Data Dictionary.csv
Demographics\demographics.csv
EMA\covid_ema.csv
EMA\Data Dictionary (COVID EMA).csv
EMA\Data Dictionary (general EMA).csv
EMA\general_ema.csv
Raw Sensing\call_log\calllog.csv
Raw Sensing\call_log\description.txt
Raw Sensing\running_apps\0ba15aa0582c5e825710d42fe3eb231d.csv
Raw Sensing\running_apps\169d3212668c4eed434dafd32e33c946.csv
Raw Sensing\running_apps\17fb1517cc9d7e416782bffb8edd9070.csv
Raw Sensing\running_apps\1d2263527eed2a54e88d340fb8e55308.csv
Raw Sensing\running_apps\212521733ea8eeff63d997dc0213c692.csv
Raw Sensing\running_apps\3528757eaf1d15ee6c778eb79cd02de7.csv
Raw Sensing\running_apps\35cf1abf179310dc33907d953f590366.csv
Raw Sensing\running_apps\45555d3305e255895b11695c179a98cd.csv
Raw Sensing\running_apps\557b44c8daf2ddb5d02313f20f2505b9.csv
Raw Sensing\running_apps\5e1d4d2718012c036ea65bfe3c80c0e7.csv
Raw Sensing\running_apps\73e13f8273906f7f43a077f95ec48e7d.csv
Raw Sensing\running_apps\7d2c632a05bbb03ca97555d61be

In [5]:
inventory = pd.DataFrame({
    "file_name": [f.name for f in all_files],
    "relative_path": [str(f.relative_to(DATA_DIR)) for f in all_files],
    "suffix": [f.suffix.lower() for f in all_files],
    "size_bytes": [f.stat().st_size for f in all_files],
})

inventory["path_parts"] = inventory["relative_path"].apply(lambda x: x.replace("\\", "/").split("/"))
inventory["top_level_group"] = inventory["path_parts"].apply(lambda x: x[0] if len(x) > 0 else None)
inventory["subgroup"] = inventory["path_parts"].apply(lambda x: x[1] if len(x) > 1 else None)

inventory.head()

,file_name,relative_path,suffix,size_bytes,path_parts,top_level_group,subgroup
0,Data Dictionary.csv,Demographics\Data Dictionary.csv,.csv,133,"[Demographics, Data Dictionary.csv]",Demographics,Data Dictionary.csv
1,demographics.csv,Demographics\demographics.csv,.csv,9088,"[Demographics, demographics.csv]",Demographics,demographics.csv
2,covid_ema.csv,EMA\covid_ema.csv,.csv,1027081,"[EMA, covid_ema.csv]",EMA,covid_ema.csv
3,Data Dictionary (COVID EMA).csv,EMA\Data Dictionary (COVID EMA).csv,.csv,1489,"[EMA, Data Dictionary (COVID EMA).csv]",EMA,Data Dictionary (COVID EMA).csv
4,Data Dictionary (general EMA).csv,EMA\Data Dictionary (general EMA).csv,.csv,2890,"[EMA, Data Dictionary (general EMA).csv]",EMA,Data Dictionary (general EMA).csv


In [6]:
suffix_counts = (
    inventory.groupby("suffix")
    .size()
    .reset_index(name="file_count")
    .sort_values("file_count", ascending=False)
)

suffix_counts

,suffix,file_count
0,.csv,251
2,.txt,4
1,.md,1


In [7]:
output_tables = Path("../../outputs/tables")
output_tables.mkdir(parents=True, exist_ok=True)

suffix_counts.to_csv(output_tables / "file_counts_by_suffix.csv", index=False)

In [8]:
def guess_role(relative_path: str) -> str:
    path_lower = relative_path.lower()
    
    if "dictionary" in path_lower:
        return "dictionary"
    if "ema" in path_lower:
        return "label_or_survey"
    if "demographics" in path_lower:
        return "metadata"
    if "raw sensing" in path_lower:
        return "sensor"
    return "unknown"

inventory["role_guess"] = inventory["relative_path"].apply(guess_role)
inventory = inventory.drop(columns=["path_parts"], errors="ignore")

inventory.sort_values(["top_level_group", "subgroup", "file_name"]).head(50)

,file_name,relative_path,suffix,size_bytes,top_level_group,subgroup,role_guess
0,Data Dictionary.csv,Demographics\Data Dictionary.csv,.csv,133,Demographics,Data Dictionary.csv,dictionary
1,demographics.csv,Demographics\demographics.csv,.csv,9088,Demographics,demographics.csv,metadata
3,Data Dictionary (COVID EMA).csv,EMA\Data Dictionary (COVID EMA).csv,.csv,1489,EMA,Data Dictionary (COVID EMA).csv,dictionary
4,Data Dictionary (general EMA).csv,EMA\Data Dictionary (general EMA).csv,.csv,2890,EMA,Data Dictionary (general EMA).csv,dictionary
2,covid_ema.csv,EMA\covid_ema.csv,.csv,1027081,EMA,covid_ema.csv,label_or_survey
5,general_ema.csv,EMA\general_ema.csv,.csv,16449907,EMA,general_ema.csv,label_or_survey
6,calllog.csv,Raw Sensing\call_log\calllog.csv,.csv,16109148,Raw Sensing,call_log,sensor
7,description.txt,Raw Sensing\call_log\description.txt,.txt,404,Raw Sensing,call_log,sensor
8,0ba15aa0582c5e825710d42fe3eb231d.csv,Raw Sensing\running_apps\0ba15aa0582c5e825710d...,.csv,8977666,Raw Sensing,running_apps,sensor
9,169d3212668c4eed434dafd32e33c946.csv,Raw Sensing\running_apps\169d3212668c4eed434da...,.csv,2645190,Raw Sensing,running_apps,sensor


In [9]:
role_counts = (
    inventory.groupby("role_guess")
    .size()
    .reset_index(name="file_count")
    .sort_values("file_count", ascending=False)
)

role_counts

,role_guess,file_count
3,sensor,245
0,dictionary,5
4,unknown,3
1,label_or_survey,2
2,metadata,1


In [10]:
inventory.to_csv(output_tables / "file_inventory.csv", index=False)
role_counts.to_csv(output_tables / "file_counts_by_role.csv", index=False)

---
## Column Inventory

Loads the key data files and records every column name, dtype, null count,
and sample value. Covers demographics, EMA files, and sensing.
Saved to outputs/tables/column_inventory.csv.

In [11]:
def load_table(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

In [12]:
important_files = [
    DATA_DIR / "Demographics" / "demographics.csv",
    DATA_DIR / "Demographics" / "Data Dictionary.csv",
    DATA_DIR / "EMA" / "general_ema.csv",
    DATA_DIR / "EMA" / "Data Dictionary (general EMA).csv",
    DATA_DIR / "EMA" / "covid_ema.csv",
    DATA_DIR / "EMA" / "Data Dictionary (COVID EMA).csv",
    DATA_DIR / "Sensing" / "sensing.csv",
]

In [13]:
column_inventory_rows = []

for path in important_files:
    if not path.exists():
        print(f"Missing: {path}")
        continue
        
    df = load_table(path)
    
    for col in df.columns:
        series = df[col]
        column_inventory_rows.append({
            "file_name": path.name,
            "relative_path": str(path.relative_to(DATA_DIR)),
            "column_name": col,
            "dtype": str(series.dtype),
            "null_count": int(series.isna().sum()),
            "null_pct": float(series.isna().mean()),
            "n_unique": int(series.nunique(dropna=True)),
            "sample_value_1": str(series.dropna().iloc[0]) if series.dropna().shape[0] > 0 else None,
        })

column_inventory = pd.DataFrame(column_inventory_rows)
column_inventory.head(30)

,file_name,relative_path,column_name,dtype,null_count,null_pct,n_unique,sample_value_1
0,demographics.csv,Demographics\demographics.csv,uid,str,0,0.000000,216,3569e2f520db9014b4acc4227a6421c1
1,demographics.csv,Demographics\demographics.csv,gender,str,0,0.000000,3,both
2,demographics.csv,Demographics\demographics.csv,race,str,0,0.000000,8,white
3,Data Dictionary.csv,Demographics\Data Dictionary.csv,Variable,str,0,0.000000,3,uid
4,Data Dictionary.csv,Demographics\Data Dictionary.csv,Definition,str,0,0.000000,3,Unique user ID
5,general_ema.csv,EMA\general_ema.csv,uid,str,0,0.000000,220,1ff6d7f34acb354430e7323a35ff7703
6,general_ema.csv,EMA\general_ema.csv,day,int64,0,0.000000,1762,20170907
7,general_ema.csv,EMA\general_ema.csv,pam,float64,181969,0.837968,16,7.0
8,general_ema.csv,EMA\general_ema.csv,phq4-1,float64,181807,0.837222,4,2.0
9,general_ema.csv,EMA\general_ema.csv,phq4-2,float64,181807,0.837222,4,2.0


In [14]:
column_inventory.to_csv(output_tables / "column_inventory.csv", index=False)

---
## Student Count Investigation

Fresh analysis of unique student IDs across all dataset files.
No assumptions about expected counts, exclusions, or cohort sizes.
Everything is derived from the data itself.

Files examined:
- EMA/general_ema.csv
- EMA/covid_ema.csv
- Sensing/sensing.csv
- Demographics/demographics.csv

In [15]:
# Load raw files with no filtering, no exclusions, no preprocessing
raw_ema   = pd.read_csv(DATA_DIR / "EMA" / "general_ema.csv")
raw_covid = pd.read_csv(DATA_DIR / "EMA" / "covid_ema.csv")
raw_sens  = pd.read_csv(DATA_DIR / "Sensing" / "sensing.csv")
raw_demo  = pd.read_csv(DATA_DIR / "Demographics" / "demographics.csv")

print("Raw file shapes (rows x columns):")
print(f"  general_ema  : {raw_ema.shape}")
print(f"  covid_ema    : {raw_covid.shape}")
print(f"  sensing      : {raw_sens.shape}")
print(f"  demographics : {raw_demo.shape}")

Raw file shapes (rows x columns):
  general_ema  : (217155, 19)
  covid_ema    : (16511, 12)
  sensing      : (216065, 651)
  demographics : (216, 3)


In [16]:
uids_ema   = set(raw_ema["uid"].unique())
uids_covid = set(raw_covid["uid"].unique())
uids_sens  = set(raw_sens["uid"].unique())
uids_demo  = set(raw_demo["uid"].unique())

print("Unique UIDs per file:")
print(f"  general_ema  : {len(uids_ema)}")
print(f"  covid_ema    : {len(uids_covid)}")
print(f"  sensing      : {len(uids_sens)}")
print(f"  demographics : {len(uids_demo)}")

Unique UIDs per file:
  general_ema  : 220
  covid_ema    : 181
  sensing      : 220
  demographics : 216


In [17]:
in_all_four       = uids_ema & uids_covid & uids_sens & uids_demo
in_ema_sens_demo  = uids_ema & uids_sens & uids_demo
in_ema_and_sens   = uids_ema & uids_sens
in_ema_only       = uids_ema - uids_covid - uids_sens - uids_demo
in_demo_only      = uids_demo - uids_ema - uids_covid - uids_sens
in_sens_not_ema   = uids_sens - uids_ema
in_ema_not_demo   = uids_ema - uids_demo
in_demo_not_ema   = uids_demo - uids_ema

print("Set membership analysis:")
print(f"  In all four files             : {len(in_all_four)}")
print(f"  In EMA + sensing + demo       : {len(in_ema_sens_demo)}")
print(f"  In EMA + sensing              : {len(in_ema_and_sens)}")
print(f"  In EMA only (not in others)   : {len(in_ema_only)}")
print(f"  In demographics only          : {len(in_demo_only)}")
print(f"  In sensing but not EMA        : {len(in_sens_not_ema)}")
print(f"  In EMA but not demographics   : {len(in_ema_not_demo)}")
print(f"  In demographics but not EMA   : {len(in_demo_not_ema)}")

Set membership analysis:
  In all four files             : 180
  In EMA + sensing + demo       : 216
  In EMA + sensing              : 220
  In EMA only (not in others)   : 0
  In demographics only          : 0
  In sensing but not EMA        : 0
  In EMA but not demographics   : 4
  In demographics but not EMA   : 0


In [18]:
# Any UID that appears in some files but not others is anomalous
# Print details for every such student
def describe_ema_student(uid, df):
    rows = df[df["uid"] == uid]
    response_cols = ["phq4-1","phq4-2","phq4-3","phq4-4",
                     "sse3-1","sse3-2","sse3-3","sse3-4",
                     "phq4_score","stress","social_level"]
    existing = [c for c in response_cols if c in df.columns]
    has_resp = rows[existing].notna().any(axis=1)
    return {
        "total_rows":        len(rows),
        "completed_surveys": int(has_resp.sum()),
        "first_day":         rows["day"].min(),
        "last_day":          rows["day"].max(),
        "span_days":         len(rows["day"].unique()),
    }

# UIDs in sensing but not in EMA
if in_sens_not_ema:
    print(f"UIDs in sensing but NOT in EMA ({len(in_sens_not_ema)}):")
    for uid in sorted(in_sens_not_ema):
        rows = raw_sens[raw_sens["uid"] == uid]
        print(f"  {uid} | sensing rows: {len(rows)} | "
              f"days: {rows['day'].min()} to {rows['day'].max()}")
else:
    print("No UIDs in sensing but not in EMA.")

print()

# UIDs in demographics but not in EMA
if in_demo_not_ema:
    print(f"UIDs in demographics but NOT in EMA ({len(in_demo_not_ema)}):")
    for uid in sorted(in_demo_not_ema):
        demo_row = raw_demo[raw_demo["uid"] == uid]
        print(f"  {uid}")
        print(f"    Demographics: {demo_row.to_dict('records')}")
else:
    print("No UIDs in demographics but not in EMA.")

print()

# UIDs in EMA but not in demographics
if in_ema_not_demo:
    print(f"UIDs in EMA but NOT in demographics ({len(in_ema_not_demo)}):")
    for uid in sorted(in_ema_not_demo):
        info = describe_ema_student(uid, raw_ema)
        print(f"  {uid} | rows: {info['total_rows']} | "
              f"surveys: {info['completed_surveys']} | "
              f"days: {info['first_day']} to {info['last_day']}")
else:
    print("No UIDs in EMA but not in demographics.")

No UIDs in sensing but not in EMA.

No UIDs in demographics but not in EMA.

UIDs in EMA but NOT in demographics (4):
  8617ddac1f48b148e3683738519b2e7a | rows: 1269 | surveys: 155 | days: 20180925 to 20220704
  df5e798581def8d477316520953b9171 | rows: 2 | surveys: 0 | days: 20171002 to 20171009
  e6d71fe4a3c10b075ae1cf51a2fe6cfd | rows: 4 | surveys: 0 | days: 20181022 to 20181025
  ea716dd032aaa0dcf8bfa36b1811917f | rows: 50 | surveys: 9 | days: 20170907 to 20171026


In [19]:
# For every unique UID in general_ema, count how many rows they have
# and how many are completed surveys
response_cols = ["phq4-1","phq4-2","phq4-3","phq4-4",
                 "sse3-1","sse3-2","sse3-3","sse3-4",
                 "phq4_score","stress","social_level"]
existing_resp = [c for c in response_cols if c in raw_ema.columns]
raw_ema["_has_response"] = raw_ema[existing_resp].notna().any(axis=1)

per_student_ema = (
    raw_ema.groupby("uid")
    .agg(
        total_rows=("uid", "count"),
        completed_surveys=("_has_response", "sum"),
        first_day=("day", "min"),
        last_day=("day", "max"),
    )
    .reset_index()
)
per_student_ema["span_days"] = (
    pd.to_datetime(per_student_ema["last_day"].astype(str), format="%Y%m%d") -
    pd.to_datetime(per_student_ema["first_day"].astype(str), format="%Y%m%d")
).dt.days + 1

print(f"Total unique students in general_ema: {len(per_student_ema)}")
print(f"\nCompleted survey count distribution:")
print(per_student_ema["completed_surveys"].describe().round(1))

# Flag students with very few completed surveys
print(f"\nStudents with 0 completed surveys:")
zero = per_student_ema[per_student_ema["completed_surveys"] == 0]
print(zero[["uid","total_rows","completed_surveys","first_day","last_day","span_days"]].to_string(index=False))

print(f"\nStudents with 1-4 completed surveys:")
few = per_student_ema[
    (per_student_ema["completed_surveys"] > 0) &
    (per_student_ema["completed_surveys"] < 5)
]
print(few[["uid","total_rows","completed_surveys","first_day","last_day","span_days"]].to_string(index=False))

Total unique students in general_ema: 220

Completed survey count distribution:
count    220.0
mean     160.7
std       88.4
min        0.0
25%       94.0
50%      168.5
75%      205.0
max      441.0
Name: completed_surveys, dtype: float64

Students with 0 completed surveys:
                             uid  total_rows  completed_surveys  first_day  last_day  span_days
df5e798581def8d477316520953b9171           2                  0   20171002  20171009          8
e6d71fe4a3c10b075ae1cf51a2fe6cfd           4                  0   20181022  20181025          4

Students with 1-4 completed surveys:
                             uid  total_rows  completed_surveys  first_day  last_day  span_days
ad15fc229da933fbf1fc0f92fc9b55a3           2                  1   20180923  20180924          2


In [20]:
# How many sensing rows does each UID have, and what is their date range
per_student_sens = (
    raw_sens.groupby("uid")
    .agg(
        total_rows=("uid", "count"),
        first_day=("day", "min"),
        last_day=("day", "max"),
    )
    .reset_index()
)

print(f"Total unique students in sensing: {len(per_student_sens)}")
print(f"\nRow count distribution:")
print(per_student_sens["total_rows"].describe().round(1))

# Flag students with very few sensing rows
few_sens = per_student_sens[per_student_sens["total_rows"] < 10]
if not few_sens.empty:
    print(f"\nStudents with fewer than 10 sensing rows:")
    print(few_sens.to_string(index=False))
else:
    print("\nNo students with fewer than 10 sensing rows.")

Total unique students in sensing: 220

Row count distribution:
count     220.0
mean      982.1
std       381.5
min         2.0
25%       749.0
50%      1143.5
75%      1280.8
max      1370.0
Name: total_rows, dtype: float64

Students with fewer than 10 sensing rows:
                             uid  total_rows  first_day  last_day
ad15fc229da933fbf1fc0f92fc9b55a3           2   20180923  20180924
df5e798581def8d477316520953b9171           2   20171002  20171009
e6d71fe4a3c10b075ae1cf51a2fe6cfd           4   20181022  20181025


In [21]:
# Build a matrix showing which files each UID appears in
all_uids = uids_ema | uids_covid | uids_sens | uids_demo

presence = pd.DataFrame({
    "uid":         list(all_uids),
    "in_ema":      [uid in uids_ema   for uid in all_uids],
    "in_covid":    [uid in uids_covid for uid in all_uids],
    "in_sensing":  [uid in uids_sens  for uid in all_uids],
    "in_demo":     [uid in uids_demo  for uid in all_uids],
})
presence["files_count"] = (
    presence[["in_ema","in_covid","in_sensing","in_demo"]]
    .sum(axis=1)
)

print("Presence matrix summary:")
print(presence.groupby(
    ["in_ema","in_covid","in_sensing","in_demo"]
).size().reset_index(name="n_students").to_string(index=False))

print(f"\nUIDs present in all 4 files    : {(presence['files_count']==4).sum()}")
print(f"UIDs present in exactly 3 files : {(presence['files_count']==3).sum()}")
print(f"UIDs present in exactly 2 files : {(presence['files_count']==2).sum()}")
print(f"UIDs present in exactly 1 file  : {(presence['files_count']==1).sum()}")

# Print the anomalous ones (not in all files)
anomalous = presence[presence["files_count"] < 4]
if not anomalous.empty:
    print(f"\nUIDs NOT present in all four files ({len(anomalous)}):")
    print(anomalous.to_string(index=False))

Presence matrix summary:
 in_ema  in_covid  in_sensing  in_demo  n_students
   True     False        True    False           3
   True     False        True     True          36
   True      True        True    False           1
   True      True        True     True         180

UIDs present in all 4 files    : 180
UIDs present in exactly 3 files : 37
UIDs present in exactly 2 files : 3
UIDs present in exactly 1 file  : 0

UIDs NOT present in all four files (40):
                             uid  in_ema  in_covid  in_sensing  in_demo  files_count
07eab8e219b6a4dcd45167dc216e1189    True     False        True     True            3
c0b0998fe60a905081764378d1102494    True     False        True     True            3
76b564dca3ab4d899e5a30d3cbcb0aac    True     False        True     True            3
3528757eaf1d15ee6c778eb79cd02de7    True     False        True     True            3
ac70fe1f8115ac361f2023269c011c3e    True     False        True     True            3
596df3c263e97eb09d44a

In [22]:
# For each UID that is anomalous in any way, describe why
# This produces the definitive exclusion decision table
decisions = []

for uid in sorted(all_uids):
    in_ema     = uid in uids_ema
    in_covid   = uid in uids_covid
    in_sensing = uid in uids_sens
    in_demo    = uid in uids_demo

    ema_info = describe_ema_student(uid, raw_ema) if in_ema else {}
    total     = ema_info.get("total_rows", 0)
    surveys   = ema_info.get("completed_surveys", 0)
    span      = ema_info.get("span_days", 0)

    # Classify reason
    if not in_ema:
        reason = "not_in_ema"
    elif not in_sensing:
        reason = "not_in_sensing"
    elif not in_demo:
        reason = "not_in_demographics"
    elif surveys == 0:
        reason = "zero_completed_surveys"
    elif surveys < 5:
        reason = f"very_few_surveys_{surveys}"
    elif span < 30:
        reason = f"very_short_span_{span}_days"
    else:
        reason = "ok"

    if reason != "ok":
        decisions.append({
            "uid":         uid,
            "in_ema":      in_ema,
            "in_covid":    in_covid,
            "in_sensing":  in_sensing,
            "in_demo":     in_demo,
            "total_rows":  total,
            "surveys":     surveys,
            "span_days":   span,
            "reason":      reason,
        })

decisions_df = pd.DataFrame(decisions)
print(f"UIDs flagged as anomalous: {len(decisions_df)}")
if not decisions_df.empty:
    print(decisions_df.to_string(index=False))

UIDs flagged as anomalous: 5
                             uid  in_ema  in_covid  in_sensing  in_demo  total_rows  surveys  span_days              reason
8617ddac1f48b148e3683738519b2e7a    True      True        True    False        1269      155       1269 not_in_demographics
ad15fc229da933fbf1fc0f92fc9b55a3    True     False        True     True           2        1          2  very_few_surveys_1
df5e798581def8d477316520953b9171    True     False        True    False           2        0          2 not_in_demographics
e6d71fe4a3c10b075ae1cf51a2fe6cfd    True     False        True    False           4        0          4 not_in_demographics
ea716dd032aaa0dcf8bfa36b1811917f    True     False        True    False          50        9         50 not_in_demographics


In [23]:
# Derive final usable population from evidence, not assumptions
ok_uids = all_uids - set(decisions_df["uid"].tolist())

print("=" * 55)
print("STUDENT COUNT SUMMARY - DERIVED FROM DATA")
print("=" * 55)
print(f"  Total unique UIDs across all files : {len(all_uids)}")
print(f"  UIDs in general_ema                : {len(uids_ema)}")
print(f"  UIDs in sensing                    : {len(uids_sens)}")
print(f"  UIDs in covid_ema                  : {len(uids_covid)}")
print(f"  UIDs in demographics               : {len(uids_demo)}")
print()
print(f"  Flagged as anomalous               : {len(decisions_df)}")
print(f"  Usable population (ok UIDs)        : {len(ok_uids)}")
print()
print("Anomalous UIDs breakdown:")
if not decisions_df.empty:
    for _, row in decisions_df.iterrows():
        print(f"  {row['uid'][:8]}...  reason: {row['reason']}")

# Save the decisions table
decisions_df.to_csv(
    "../../outputs/tables/uid_exclusion_decisions.csv",
    index=False
)
print("\nSaved to outputs/tables/uid_exclusion_decisions.csv")

STUDENT COUNT SUMMARY - DERIVED FROM DATA
  Total unique UIDs across all files : 220
  UIDs in general_ema                : 220
  UIDs in sensing                    : 220
  UIDs in covid_ema                  : 181
  UIDs in demographics               : 216

  Flagged as anomalous               : 5
  Usable population (ok UIDs)        : 215

Anomalous UIDs breakdown:
  8617ddac...  reason: not_in_demographics
  ad15fc22...  reason: very_few_surveys_1
  df5e7985...  reason: not_in_demographics
  e6d71fe4...  reason: not_in_demographics
  ea716dd0...  reason: not_in_demographics

Saved to outputs/tables/uid_exclusion_decisions.csv
